# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR2 (Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
# Get metadata object (do not treat as dict or iterate)
metadata = dataset.metadata

# Print dataset basic info (using direct attributes)
print(f"{metadata.name}: {metadata.description}")
print(f"Version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Identifier: {metadata.identifier}")
print(f"Date Published: {metadata.datePublished}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Croissant schemas organize data in record sets, fields, and columns. We'll discover all top-level record sets and fields by their `@id`.

In [ ]:
# List all record sets
record_sets = dataset.record_sets()
print("Record sets found in the dataset:")
for rs in record_sets:
    print(f"- @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For each record set, list its fields (columns)
print("\nFields for each record set:")
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']} ({rs.get('name', '')})")
    fields = dataset.fields(rs['@id'])
    for field in fields:
        print(f"  - Field @id: {field['@id']} | label: {field.get('label','')} | dataType: {field.get('dataType','')}")

### Preview First Records
Let's preview a few records from the main record set using its `@id`.

In [ ]:
# Pick the main record set (usually the first or the primary one)
main_record_set_id = record_sets[0]['@id'] if record_sets else None
print(f"Previewing sample records from Record Set: {main_record_set_id}")

# Show a few records
for i, record in enumerate(dataset.records(record_set=main_record_set_id)):
    print(record)
    if i >= 2:
        break

## 3. Data Extraction
Load data from all record sets into DataFrames for analysis. Each record set and field is referenced by its `@id`.

In [ ]:
# Extract data from each record set into pandas DataFrames
dataframes = {}
for rs in record_sets:
    rs_id = rs['@id']
    records = list(dataset.records(record_set=rs_id))
    # Only add if records exist
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"RecordSet '@id': {rs_id}")
        print(f"Columns: {dataframes[rs_id].columns.tolist()}")
        display(dataframes[rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data.

We'll demonstrate filtering and analyzing a numeric field (e.g., age at diagnosis) and grouping by a categorical field (e.g., MSI status or anatomical location).

In [ ]:
# Pick the primary record set DataFrame
primary_rs_id = main_record_set_id
df = dataframes[primary_rs_id]

# Determine a numeric field to analyze
# We extract the field ids for numeric (Integer or Float) fields
numeric_field_id = None
fields = dataset.fields(primary_rs_id)
for f in fields:
    if f.get('dataType', '').lower() in ['integer', 'float', 'number']:
        numeric_field_id = f['@id']
        break

print(f"Selected numeric field: {numeric_field_id}")

# If no numeric field found, skip EDA
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Pick the first categorical field for grouping
    group_field_id = None
    for f in fields:
        if f.get('dataType', '').lower() not in ['integer', 'float', 'number'] and f['@id'] in df.columns:
            group_field_id = f['@id']
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot the distribution of the numeric field and its normalized values.

In [ ]:
# Visualization of numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if f"{numeric_field_id}_normalized" in filtered_df.columns:
        plt.figure(figsize=(8,4))
        sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), kde=True)
        plt.title(f"Normalized Distribution of {numeric_field_id} (filtered)")
        plt.xlabel(f"{numeric_field_id}_normalized")
        plt.show()

## 6. Conclusion
This notebook demonstrated loading and processing the FAIR2 colorectal cancer dataset using `mlcroissant`, referencing all entities by their `@id`.

- Loaded dataset metadata and records.
- Reviewed available record sets and fields using their `@id`.
- Extracted data to DataFrames and performed basic EDA on numeric and categorical fields.
- Visualized data distributions.

The FAIR2 dataset enables reproducible clinical oncology research, supporting analysis of anatomical, molecular, and clinicopathological predictors of second primary colorectal cancer in cancer survivors.